In [ ]:
# https://www.aluracursos.com/blog/langgraph-que-es-como-usarlo-y-sus-funcionalidades
# Instalar bibliotecas necesarias
!pip install -q -U langgraph langchain-openai

In [ ]:
!pip show langgraph

In [ ]:
import os
# Permite usar los secrets
from google.colab import userdata
# Obteniendo la llave de OpenAI de forma segura
os.environ['OPENAI_API_KEY']=userdata.get('OPENAI_API_KEY')

In [ ]:
# Proyecto LangGraph
# Importaciones principales
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
# Definición del estado global
class Estado(TypedDict):
    pregunta: str
    respuesta: str
# Iniciando el modelo LLM
modelo = ChatOpenAI(model="gpt-4o-mini")
# Nodo 1: Recibe y muestra la pregunta
def nodo_recibir_pregunta(state: Estado):
    print("🔹 Pregunta recibida:", state["pregunta"])
    return state
# Nodo 2: Genera la respuesta vía LLM y la guarda en el estado
def nodo_generar_respuesta(state: Estado):
    pregunta = state["pregunta"]
    respuesta = modelo.invoke(pregunta)  # Llamada al LLM
    state["respuesta"] = respuesta.content
    print("🔹 Respuesta generada con éxito.")
    return state
# Construyendo el grafo
grafo = StateGraph(Estado)
grafo.add_node("recibir_pregunta", nodo_recibir_pregunta)
grafo.add_node("generar_respuesta", nodo_generar_respuesta)
grafo.add_edge("recibir_pregunta", "generar_respuesta")
grafo.add_edge("generar_respuesta", END)
# Definiendo el inicio del flujo
grafo.set_entry_point("recibir_pregunta")
# Compila el grafo
app = grafo.compile()

In [ ]:
from IPython.display import Image, display

graph = app.get_graph()

try:
    display(Image(graph.draw_mermaid_png(max_retries=3, retry_delay=1.0)))
except Exception:
    try:
        print(graph.draw_ascii())
    except ImportError:
        print("Diagrama Mermaid (copia en https://mermaid.live si quieres verlo como imagen):\n")
        print(graph.draw_mermaid())


In [ ]:
# Ejecutando el grafo
estado_inicial = {
    "pregunta": "Explica qué es LangGraph de forma sencilla.",
    "respuesta": ""
}
resultado = app.invoke(estado_inicial)
print("\n===== Respuesta del agente =====")
print(resultado["respuesta"])